# 🚀 StreamCI Quickstart — PyStreamCI Client Library

This notebook walks you through the **core data lifecycle** in StreamCI using the `pystreamci` Python client library:

1. **Connect** to the StreamCI server
2. **Insert** data (single, batch, file upload)
3. **Query** data (filter, project, sort, limit)
4. **Analyze & Visualize** with Pandas and Matplotlib
5. **Simulate** real-time streaming ingestion

> 📦 Install the library first:  
> ```bash
> pip install pystreamci
> ```

---
## 1. Connect to StreamCI

Use `pystreamci.connect()` to create a client. The recommended pattern is a **context manager** (`with` statement), which automatically opens and closes the connection.

You need:
- **`host`** — the StreamCI API endpoint
- **`target`** — your database/app name
- **`secret_key`** — your API secret key (or use `username`/`password` for password auth)

In [ ]:
import pystreamci

# ✏️ Fill in your credentials below
HOST       = "https://api.streamci.org"
TARGET     = "userXX"          # Replace with your target name
SECRET_KEY = ""                # Replace with your secret key

In [ ]:
# Quick connection test
with pystreamci.connect(HOST, TARGET, secret_key=SECRET_KEY) as client:
    print("✅ Connected:", client)

---
## 2. Insert a Single Document

`client.insert(data)` sends one record to StreamCI.

- Timestamps must be in the `{"$date": "ISO-8601"}` format.  
- Use `pystreamci.make_date()` to convert a `datetime` object or ISO string automatically.

In [ ]:
import random

with pystreamci.connect(HOST, TARGET, secret_key=SECRET_KEY) as client:
    result = client.insert({
        "time":     pystreamci.make_date("2025-08-04T10:00:00.001Z"),
        "deviceID": "sensor1",
        "val1":     random.randint(0, 50),
        "val2":     round(random.uniform(0, 25.0), 2),
    })
    print("Response:", result)

---
## 3. Insert Many Documents (Bulk Load)

`client.insert_many(records)` sends a **batch of records** in one request.  
Use `batch_size` to control how many records are sent per request (default: 1000, max: 10,000).

> ⚠️ **Tip:** `insert_many` is best for small-to-medium batches (up to a few thousand records).  
> For larger datasets, use `insert_file()` shown in the next section.

In [ ]:
records = [
    {
        "time":     pystreamci.make_date("2025-08-04T10:01:00.001Z"),
        "deviceID": "sensor1",
        "val1":     random.randint(0, 50),
        "val2":     round(random.uniform(0, 25.0), 2),
    },
    {
        "time":     pystreamci.make_date("2025-08-04T10:02:00.001Z"),
        "deviceID": "sensor2",
        "val1":     random.randint(40, 80),
        "val2":     round(random.uniform(20, 75.0), 2),
    },
    {
        "time":     pystreamci.make_date("2025-08-04T10:03:00.001Z"),
        "deviceID": "sensor3",
        "val1":     random.randint(70, 100),
        "val2":     round(random.uniform(70, 100.0), 2),
    },
]

with pystreamci.connect(HOST, TARGET, secret_key=SECRET_KEY) as client:
    results = client.insert_many(records, batch_size=1000)
    print("Responses:", results)

---
## 4. Upload a File (NDJSON / CSV)

For **large datasets**, use `client.insert_file()` to upload a file directly.  
Supported formats: **NDJSON** (Newline-Delimited JSON) and **CSV**.

Let's first preview the NDJSON file, then upload it.

In [ ]:
import json

file_path = "data.ndjson"
count = 0

# Preview the first 5 records
with open(file_path, "r") as f:
    for line in f:
        if line.strip():
            if count < 5:
                print(json.loads(line))
            count += 1

print(f"\nTotal records in file: {count}")

In [ ]:
# Upload the NDJSON file in one call
with pystreamci.connect(HOST, TARGET, secret_key=SECRET_KEY) as client:
    result = client.insert_file("data.ndjson")
    print("Upload result:", result)

---
## 5. Query Data

`client.query()` retrieves documents and returns a **list of dicts**.  
You can use the following options:

| Parameter  | Description | Example |
|------------|-------------|--------|
| `query`    | Filter criteria (MongoDB-style) | `{"deviceID": "sensor1"}` |
| `project`  | Fields to include | `["time", "val1"]` |
| `sort`     | Sort order (`1`=asc, `-1`=desc) | `{"time": -1}` |
| `limit`    | Max records to return | `100` |

In [ ]:
with pystreamci.connect(HOST, TARGET, secret_key=SECRET_KEY) as client:

    # 5-1. Query all records
    docs = client.query({})
    print(f"All records: {len(docs)}")

In [ ]:
with pystreamci.connect(HOST, TARGET, secret_key=SECRET_KEY) as client:

    # 5-2. Filter by time range
    docs = client.query({
        "time": {
            "$gte": pystreamci.make_date("2025-08-04T12:00:00.000Z"),
            "$lt":  pystreamci.make_date("2025-08-04T13:00:00.000Z"),
        }
    })
    print(f"Records in 12:00–13:00 UTC: {len(docs)}")

In [ ]:
with pystreamci.connect(HOST, TARGET, secret_key=SECRET_KEY) as client:

    # 5-3. Filter by time range + deviceID
    docs = client.query({
        "time": {
            "$gte": pystreamci.make_date("2025-08-04T12:00:00.000Z"),
            "$lt":  pystreamci.make_date("2025-08-04T13:00:00.000Z"),
        },
        "deviceID": "sensor1",
    })
    print(f"sensor1 in 12:00–13:00: {len(docs)} records")
    if docs:
        print("Sample:", docs[0])

In [ ]:
with pystreamci.connect(HOST, TARGET, secret_key=SECRET_KEY) as client:

    # 5-4. Projection — select specific fields only
    docs = client.query(
        {"deviceID": "sensor1"},
        project=["time", "deviceID", "val1"],
    )
    print(f"Projected results: {len(docs)} records")
    if docs:
        print("Sample:", docs[0])

In [ ]:
with pystreamci.connect(HOST, TARGET, secret_key=SECRET_KEY) as client:

    # 5-5. Sort (descending by time) + Limit
    docs = client.query(
        {"deviceID": "sensor1"},
        project=["time", "deviceID", "val1"],
        sort={"time": -1},
        limit=1,
    )
    print(f"Latest sensor1 record: {docs[0]}")

---
## 6. Data Analysis with Pandas

Query results are plain Python dicts — easily converted to a **Pandas DataFrame** for analysis.

In [ ]:
import pandas as pd

with pystreamci.connect(HOST, TARGET, secret_key=SECRET_KEY) as client:
    docs = client.query(
        {},
        project=["time", "deviceID", "val1", "val2"],
    )

# Convert to DataFrame
df = pd.DataFrame(docs)

# Parse time column
df["time"] = pd.to_datetime(df["time"], utc=True)
df["time"] = df["time"].dt.tz_convert("US/Eastern")

# Summary statistics per sensor
summary = df.groupby("deviceID")[["val1", "val2"]].describe()
print(summary.to_string())

---
## 7. Visualization with Matplotlib

Let's plot `val2` over time for each sensor as a scatter plot.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
for sensor in df["deviceID"].unique():
    subset = df[df["deviceID"] == sensor]
    plt.scatter(subset["time"], subset["val2"], label=sensor, s=3, alpha=0.5)

plt.xlabel("Time (US/Eastern)")
plt.ylabel("val2")
plt.title("val2 Over Time by Sensor")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

---
## 8. Streaming Simulation

Simulate a sensor sending data every few seconds using `client.insert()` in a loop.

> ⚠️ This cell runs continuously. **Interrupt the kernel** to stop it.

In [ ]:
import time
from datetime import datetime, timezone


def generate_random_data():
    """Generate a single synthetic sensor reading."""
    device_id = random.choice(["sensor1", "sensor2", "sensor3"])

    if device_id == "sensor1":
        val1 = random.randint(0, 50)
        val2 = round(random.uniform(0.0, 25.0), 2)
    elif device_id == "sensor2":
        val1 = random.randint(40, 80)
        val2 = round(random.uniform(20.0, 75.0), 2)
    else:  # sensor3
        val1 = random.randint(70, 100)
        val2 = round(random.uniform(70.0, 100.0), 2)

    return {
        "time":     pystreamci.make_date(datetime.now(timezone.utc)),
        "deviceID": device_id,
        "val1":     val1,
        "val2":     val2,
    }

In [ ]:
NUM_RECORDS = 100    # Total records to send
INTERVAL    = 3      # Seconds between each insert

with pystreamci.connect(HOST, TARGET, secret_key=SECRET_KEY) as client:
    for i in range(NUM_RECORDS):
        data = generate_random_data()
        try:
            client.insert(data)
            if (i + 1) % 10 == 0:
                print(f"Sent {i + 1}/{NUM_RECORDS} records...")
        except Exception as e:
            print(f"Error on record {i + 1}: {e}")
        time.sleep(INTERVAL)

print("✅ Streaming simulation complete!")